In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import linregress
from scipy import linalg
import networkx as nx # 用于生成复杂的异质拓扑
from scipy.stats import ortho_group
from scipy import stats

In [2]:
result_dir='result/'
N_arr=np.array([100,200,300,400,500,600])
C = 2e2
H_weight=1e-3

In [3]:
# --- Analysis & Power Law Plotting ---
def plot_power_law(data, label, ax, title, num_bins=None):
    """
    Plot power law distribution with adaptive binning.
    
    Parameters:
    - data: array-like, the data to plot
    - label: str, label for the data
    - ax: matplotlib axis, axis to plot on
    - title: str, title for the plot
    - num_bins: int or None, if None uses adaptive binning based on data size
    """
    # Adaptive binning based on data size
    if num_bins is None:
        # Use Freedman-Diaconis rule for bin width selection
        n = len(data)
        if n > 1000:
            # For large datasets, use more bins
            num_bins = int(np.sqrt(n))
        else:
            # For smaller datasets, use fewer bins
            num_bins = max(10, int(n / 50))
    
    # Ensure we have at least 5 bins and at most 50 bins
    num_bins = max(5, min(50, num_bins))
    
    # Log-spaced bins
    bins = np.logspace(np.log10(max(min(data), 1e-10)),  # Avoid log(0)
                      np.log10(max(data)), 
                      num_bins + 1)
    
    # Calculate histogram
    counts, bins = np.histogram(data, bins=bins)
    x = (bins[:-1] + bins[1:]) / 2
    y = counts / np.diff(bins)  # Probability density
    
    # Remove zeros for log-log fit
    mask = (y > 0) & (x > 0)
    x_fit, y_fit = x[mask], y[mask]
    
    # Check if we have enough points for fitting
    if len(x_fit) < 3:
        ax.loglog(x, y, 'o', label=f'{label} (insufficient data for fit)')
        ax.set_title(title)
        ax.set_xlabel('Value')
        ax.set_ylabel('Probability Density')
        ax.legend()
        return
    
    # Fit: log(y) = -tau * log(x) + C
    slope, intercept, r_value, p_value, std_err = linregress(np.log10(x_fit), np.log10(y_fit))
    
    # Plot data and fit
    ax.loglog(x, y, 'o', markersize=8, color='cyan',alpha=0.7, label=f'{label}')
    ax.loglog(x_fit, 10**intercept * x_fit**slope, '--', alpha=0.8, 
              linewidth=2,color='black', label=f'Fit: slope={slope:.2f}')
    
    # Add R² value if correlation is high
    if abs(r_value) > 0.8:
        ax.text(0.05, 0.95, f'R² = {r_value**2:.3f}', 
                transform=ax.transAxes, fontsize=9,
                verticalalignment='top')
    
    # ax.set_title(title)
    ax.set_xlabel(label,fontsize=16, fontweight='bold')
    ax.set_ylabel('Probability Density',fontsize=16, fontweight='bold')
    ax.legend(fontsize=14)
    ax.grid(True, alpha=0.3, which='both')

In [4]:
# --- Analysis & Power Law Plotting ---
def plot_power_law(data, label, ax, title, num_bins=None, color='#2E86AB'):
    """
    Plot power law distribution with adaptive binning.
    
    Parameters:
    - data: array-like, the data to plot
    - label: str, label for the data
    - ax: matplotlib axis, axis to plot on
    - title: str, title for the plot
    - num_bins: int or None, if None uses adaptive binning based on data size
    - color: str, color for the data points
    """
    # Set global style for the axis
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(1.5)
    ax.spines['bottom'].set_linewidth(1.5)
    
    # Adaptive binning based on data size
    if num_bins is None:
        n = len(data)
        if n > 1000:
            # Use Scott's rule for large datasets
            num_bins = int(np.ceil((3.5 * np.std(np.log10(data[data>0])) * n**(1/3))))
            num_bins = max(20, min(80, num_bins))
        else:
            # For smaller datasets, use Sturges' rule
            num_bins = int(np.ceil(np.log2(n) + 1))
            num_bins = max(10, min(30, num_bins))
    
    # Ensure we have reasonable number of bins
    num_bins = max(5, min(80, num_bins))
    
    # Clean data: remove zeros and negative values for log-scale
    data_clean = data[data > 0]
    if len(data_clean) == 0:
        ax.text(0.5, 0.5, 'No positive data points', 
                transform=ax.transAxes, ha='center', va='center')
        return
    
    # Log-spaced bins with protection against extreme values
    min_val = max(min(data_clean), 1e-10)
    max_val = max(data_clean)
    
    # Create logarithmically spaced bins
    bins = np.logspace(np.log10(min_val), np.log10(max_val), num_bins + 1)
    
    # Calculate histogram with density normalization
    counts, bin_edges = np.histogram(data_clean, bins=bins, density=True)
    x = (bin_edges[:-1] + bin_edges[1:]) / 2
    y = counts
    
    # Remove zeros for log-log fit
    mask = (y > 0) & (x > 0) & np.isfinite(y) & np.isfinite(x)
    x_fit, y_fit = x[mask], y[mask]
    
    # Check if we have enough points for fitting
    if len(x_fit) < 5:
        ax.loglog(x, y, 'o', markersize=7, color=color, alpha=0.6,
                 markerfacecolor='white', markeredgewidth=1.5,
                 label=f'{label} (insufficient data)')
        ax.set_xlabel(label, fontsize=14, fontweight='bold')
        ax.set_ylabel('Probability Density', fontsize=14, fontweight='bold')
        ax.grid(True, alpha=0.2, linestyle=':', linewidth=0.8)
        ax.set_axisbelow(True)
        return
    
    # Fit: log(y) = -tau * log(x) + C
    try:
        slope, intercept, r_value, p_value, std_err = linregress(
            np.log10(x_fit), np.log10(y_fit)
        )
    except:
        ax.loglog(x, y, 'o', markersize=7, color=color, alpha=0.6,
                 markerfacecolor='white', markeredgewidth=1.5,
                 label=f'{label} (fit failed)')
        return
    
    # Color palette
    fit_color = '#D64933'  # Red for fit line
    
    # Plot data points with enhanced styling
    ax.loglog(x, y, 'o', 
              markersize=8,
              color=color,
              markerfacecolor='white',
              markeredgewidth=2,
              alpha=0.8,
              label=f'{label}',
              zorder=2)
    
    # Add error bars based on Poisson statistics (sqrt(count))
    y_err = np.sqrt(counts[mask]) / np.diff(bins)[mask] if len(mask) > 0 else None
    if y_err is not None and len(y_err) > 0:
        ax.errorbar(x_fit, y_fit, 
                   yerr=y_err,
                   fmt='none',
                   ecolor=color,
                   alpha=0.3,
                   capsize=2,
                   elinewidth=0.8,
                   zorder=1)
    
    # Plot fit line with enhanced styling
    x_fit_smooth = np.logspace(np.log10(min(x_fit)), np.log10(max(x_fit)), 100)
    y_fit_smooth = 10**intercept * x_fit_smooth**slope
    
    ax.loglog(x_fit_smooth, y_fit_smooth, 
              '-',
              color=fit_color,
              linewidth=2.5,
              alpha=0.85,
              label=f'$slope = {slope:.2f}$',
              zorder=3)
    
    # Add shaded region for fit uncertainty (optional)
    # y_fit_upper = 10**(intercept + std_err) * x_fit_smooth**slope
    # y_fit_lower = 10**(intercept - std_err) * x_fit_smooth**slope
    # ax.fill_between(x_fit_smooth, y_fit_lower, y_fit_upper,
    #                 color=fit_color, alpha=0.1, zorder=0)
    
    # Add R² and p-value in a clean box
    r_squared = r_value**2
    stats_text = f'$R^2 = {r_squared:.3f}$'
    # stats_text = f'$R^2 = {r_squared:.3f}$\n$p = {p_value:.2e}$'
    
    # Place statistics in top-left corner with a clean background
    bbox_props = dict(boxstyle='round,pad=0.4', 
                     facecolor='white', 
                     alpha=0.85,
                     edgecolor='gray',
                     linewidth=0.5)
    
    ax.text(0.05, 0.95, stats_text,
            transform=ax.transAxes,
            fontsize=10,
            verticalalignment='top',
            bbox=bbox_props,
            zorder=10)
    
    # # Add power law exponent annotation
    # exponent_text = f'$\\tau = {slope:.2f} \\pm {std_err:.3f}$'
    # ax.text(0.05, 0.82, exponent_text,
    #         transform=ax.transAxes,
    #         fontsize=10,
    #         verticalalignment='top',
    #         color=fit_color,
    #         fontweight='bold',
    #         bbox=bbox_props,
    #         zorder=10)
    
    # Axis labels with enhanced formatting
    ax.set_xlabel(label, fontsize=15, fontweight='bold', labelpad=8)
    ax.set_ylabel('Probability Density', fontsize=15, fontweight='bold', labelpad=8)
    
    # Legend with improved styling
    legend = ax.legend(fontsize=12,
                      frameon=True,
                      fancybox=True,
                      shadow=True,
                      framealpha=0.95,
                      edgecolor='none',
                      loc='lower left')
    legend.get_frame().set_linewidth(0)
    
    # Grid with subtle styling
    ax.grid(True, alpha=0.25, linestyle=':', linewidth=0.8, which='both')
    ax.set_axisbelow(True)
    
    # Customize tick parameters
    ax.tick_params(axis='both', which='major', 
                   labelsize=11, width=1.5, length=6)
    ax.tick_params(axis='both', which='minor',
                   width=1, length=3)
    
    # Set axis limits with padding (MODIFIED: reduced padding on left)
    x_min, x_max = ax.get_xlim()
    y_min, y_max = ax.get_ylim()
    
    # Get actual data range
    actual_x_min = np.min(x_fit)  # Use only the fitted data range
    actual_x_max = np.max(x_fit)
    actual_y_min = np.min(y_fit)
    actual_y_max = np.max(y_fit)
    
    # Set limits with asymmetric padding (less padding on left)
    x_padding_left = actual_x_min * 0.1  # Only 10% padding on left
    x_padding_right = actual_x_max * 7  # 20% padding on right
    y_padding_bottom = actual_y_min * 0.1
    y_padding_top = actual_y_max * 7
    
    ax.set_xlim(x_padding_left, x_padding_right)
    ax.set_ylim(y_padding_bottom, y_padding_top)
    
    # Optional: Add minor ticks
    ax.minorticks_on()
    
    return slope, intercept, r_value, p_value, std_err

In [5]:
# from scipy.integrate import trapz
def simulate_quench_avalanche(A, N, num_quenches=1000):
    """
    模拟 Quench 动力学中的雪崩 (Response Avalanche)。
    核心逻辑：统计对于微小的参数变化 delta_A，系统状态变化 delta_x 的分布。
    """

    
    # 2. 初始化系统状态
    # 我们需要一个非零的“外场”或“输入” y，使得 x 有非零的平衡位置
    # Ax = y  => x = A^-1 y
    y = 0.1*np.random.randn(N)
    x_current = np.linalg.solve(A, y)
    
    avalanche_sizes = []
    # durations=[]

    # # 时间轴 (用于积分)
    # t_eval = np.logspace(1, 10, 2000) # 覆盖快慢尺度
    
    # 3. 循环执行 Quench 实验
    for _ in range(num_quenches):
        # y = 0.1*np.random.randn(N)
        # x_current = np.linalg.solve(A, y)

        
        # --- A. 施加微扰 (The Quench) ---
        # 这种扰动模拟突触权重的微小漂移或学习更新
        # 强度必须很小，否则系统会彻底崩溃



        
        perturbation_strength = 1e-6
        R = np.random.randn(N, N)
        delta_A = perturbation_strength * (R + R.T) # 对称扰动
        
        # --- B. 计算新的平衡态 ---
        A_new = A + delta_A
        
        # 检查正定性 (可选，防止数值爆炸)
        # 在微小扰动下，Sloppy 系统的软模极易变负，这里简单处理：
        # 如果奇异，则跳过统计（或视为无穷大雪崩）


        x_new = np.linalg.solve(A_new, y)
            
        # --- C. 提取雪崩规模 (Avalanche Size) ---
        # 定义：状态向量在相空间中的位移距离
        dx = x_new - x_current
        size = np.linalg.norm(dx)
        
        avalanche_sizes.append(size)

        

        


        # --- D. 更新系统 ---
        # 让系统演化下去 (Cumulative Aging) 
        # 或者重置 A (Independent Samples)
        # 这里选择 Cumulative，模拟真实系统的连续演化
        A = A_new 
        x_current = x_new

        
        # try:
        #     x_new = np.linalg.solve(A_new, y)
            
        #     # --- C. 提取雪崩规模 (Avalanche Size) ---
        #     # 定义：状态向量在相空间中的位移距离
        #     dx = x_new - x_current
        #     size = np.linalg.norm(dx)
            
        #     avalanche_sizes.append(size)

        #     # --- D. 更新系统 ---
        #     # 让系统演化下去 (Cumulative Aging) 
        #     # 或者重置 A (Independent Samples)
        #     # 这里选择 Cumulative，模拟真实系统的连续演化
        #     A = A_new 
        #     x_current = x_new

        # except np.linalg.LinAlgError:
        #     # 矩阵奇异，意味着特征值触底 0，发生了灾难性雪崩
        #     pass
        

    return np.array(avalanche_sizes)

In [ ]:
# --- 运行 ---
for N in N_arr:
    A = np.random.randn(N, N)
    # 调整迹
    current_trace = np.trace(A)
    A *= C / current_trace


    print(np.trace(A))
    
    sizes = simulate_quench_avalanche(A,N)
    
    fig,ax=plt.subplots(figsize=(6, 4.5))
    plot_power_law(sizes, r'$||\mathbf{dx}||$', ax, "Avalanche Size Distribution")
    plt.savefig(result_dir+f'dx_powerlaw_of_shuffled_A_{N}.png',bbox_inches='tight',dpi=300)